# 02 — Làm sạch, chia train/test, tăng cường dữ liệu

Luồng: **raw -> audit SNR & loại mẫu lỗi -> chia train/test (stratify theo nồng độ)
-> augment CHỈ trên train -> lưu ra `data/processed/`**

Lý do augment chỉ trên train và chia trước khi augment: nếu augment trước rồi mới
chia, các bản sao tổng hợp từ cùng 1 phổ gốc có thể rơi vào cả train lẫn test
-> rò rỉ dữ liệu (data leakage), điểm test sẽ đẹp giả tạo.

Tham số `lam` dùng trong bước audit SNR được nạp từ
`data/processed/chosen_params.json` (xuất ra từ `00_parameter_selection.ipynb`),
không hard-code lại trong notebook này để tránh lệch giữa 2 nơi.

In [ ]:
import sys, os, json
sys.path.insert(0, '../src')  # chỉnh lại nếu vị trí src/ khác

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from raman_processing import baseline_correction, calc_snr_std
from data_cleaning import build_label_table, build_snr_report, flag_low_snr_samples, summarize_flags
from augmentation import augment_dataset

os.makedirs('../data/processed', exist_ok=True)

with open('../data/processed/chosen_params.json') as f:
    params = json.load(f)
print('Tham số đã chọn (từ 00_parameter_selection.ipynb):')
print(json.dumps(params, indent=2))

AIRPLS_LAM = params['airpls_lam']

## 1. Đọc dữ liệu gốc

In [ ]:
df_raw = pd.read_excel('../data/raw/Ethanol_Methanol.xlsx')
x_full = df_raw['Raman Shift (cm-1)'].values
mask = ~np.isnan(x_full)
x = x_full[mask]
df = df_raw.loc[mask].reset_index(drop=True)

sample_cols = [c for c in df.columns if c != 'Raman Shift (cm-1)']
print(f'{len(sample_cols)} mẫu, {len(x)} điểm/phổ')

## 2. Audit chất lượng & loại mẫu SNR bất thường

Dùng `AIRPLS_LAM` vừa nạp thay vì mặc định của hàm. Ngưỡng `ratio_threshold=0.3`:
mẫu có SNR thấp hơn 30% so với median của các lần lặp cùng nồng độ sẽ bị gắn cờ.
Xem lại bảng bằng mắt trước khi loại hàng loạt.

In [ ]:
snr_report = build_snr_report(df, x, sample_cols=sample_cols,
                               baseline_method='airpls', lam=AIRPLS_LAM)
flagged = flag_low_snr_samples(snr_report, ratio_threshold=0.3)
bad = summarize_flags(flagged)

flagged.to_csv('../data/processed/snr_report.csv', index=False)
print(f'\nĐã lưu snr_report.csv ({len(flagged)} mẫu, {flagged["flagged"].sum()} bị gắn cờ)')

In [ ]:
# XÁC NHẬN THỦ CÔNG trước khi loại -- xem lại bad ở trên, sửa danh sách này nếu cần
excluded_samples = flagged.loc[flagged['flagged'], 'raw'].tolist()
print(f'Loại {len(excluded_samples)} mẫu:', excluded_samples)

clean_cols = [c for c in sample_cols if c not in excluded_samples]
df_clean = df[clean_cols].copy()
labels_clean = build_label_table(clean_cols)
labels_clean['excluded'] = False

pd.Series(excluded_samples, name='raw').to_csv('../data/processed/excluded_samples.csv', index=False)
print(f'Còn lại {len(clean_cols)} mẫu sạch để chia train/test')

## 3. Chia train / test (stratify theo nồng độ, theo PHỔ GỐC)

Chia trước khi augment. Mẫu có `concentration=None` (EMX, EMX_b) xếp riêng vì
không thể stratify theo nồng độ số -- đưa hết vào train.

In [ ]:
has_conc = labels_clean['concentration'].notna()
df_with_conc = labels_clean[has_conc].copy()
df_no_conc = labels_clean[~has_conc].copy()

strat_key = df_with_conc['series'] + '_' + df_with_conc['concentration'].astype(str)

train_raw, test_raw = train_test_split(
    df_with_conc['raw'],
    test_size=0.2,
    random_state=42,
    stratify=strat_key if strat_key.value_counts().min() >= 2 else None,
)

train_raw = pd.concat([train_raw, df_no_conc['raw']], ignore_index=True)

split_df = pd.DataFrame({
    'raw': list(train_raw) + list(test_raw),
    'split': ['train'] * len(train_raw) + ['test'] * len(test_raw),
})
split_df.to_csv('../data/processed/split_ids.csv', index=False)
print(f'Train: {len(train_raw)} mẫu | Test: {len(test_raw)} mẫu')
print(split_df.groupby('split').size())

## 4. Tăng cường dữ liệu — CHỈ áp dụng cho tập train

Test set giữ nguyên phổ thật, không augment.

In [ ]:
N_AUGMENTATIONS = 100  # chỉnh theo mục tiêu số phổ/lớp đã thống nhất

train_cols = split_df.loc[split_df['split'] == 'train', 'raw'].tolist()

aug_df, source_map = augment_dataset(
    df_clean, x, sample_cols=train_cols,
    n_augmentations=N_AUGMENTATIONS, seed=42,
)
print('Số phổ tổng hợp:', aug_df.shape[1])

## 5. Lưu kết quả ra `data/processed/`

In [ ]:
train_raw_spectra = df_clean[train_cols].copy()
test_spectra = df_clean[split_df.loc[split_df['split'] == 'test', 'raw'].tolist()].copy()

x_series = pd.Series(x, name='Raman Shift (cm-1)')

pd.concat([x_series, train_raw_spectra], axis=1).to_csv(
    '../data/processed/train_spectra_original.csv', index=False)
pd.concat([x_series, aug_df], axis=1).to_csv(
    '../data/processed/train_spectra_augmented.csv', index=False)
pd.concat([x_series, test_spectra], axis=1).to_csv(
    '../data/processed/test_spectra.csv', index=False)

pd.Series(source_map, name='source_sample').rename_axis('augmented_col').to_csv(
    '../data/processed/augmented_source_map.csv')

print('Đã lưu:')
print(' - train_spectra_original.csv  ', train_raw_spectra.shape)
print(' - train_spectra_augmented.csv ', aug_df.shape)
print(' - test_spectra.csv            ', test_spectra.shape)
print(' - augmented_source_map.csv, split_ids.csv, excluded_samples.csv, snr_report.csv')